In [1]:
import pickle

import pandas as pd
from sentence_transformers import SentenceTransformer, util
import os
import numpy as np
import json
import importlib
from pathlib import Path
import requests
from transformers import AutoModel, AutoTokenizer, AutoModelForMaskedLM
from transformers import PreTrainedTokenizer
from tensor2tensor.data_generators import text_encoder
import pickle
from sklearn.decomposition import PCA
from sklearn.manifold      import TSNE
#import umap
import tempun

from datetime import datetime, timedelta

2026-05-12 15:26:48.203638: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-12 15:26:48.245985: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
outdir = Path("../data/large_files/phylosen_instances/")

In [9]:
base_path = "/srv/models/latin-bert"

# Initialize the tokenizer with the vocab.txt file and the encoder
vocab_file_path = "/srv/models/latin-bert/vocab.txt" # "/Users/vojtechkase/Projects/latin-bert/models/latin_bert/vocab.txt"  # Update this path
subword_tokenizer_path = "/srv/models/latin-bert/latin.subword.encoder"
# Update this path
encoder = text_encoder.SubwordTextEncoder(subword_tokenizer_path)

spec = importlib.util.spec_from_file_location("latin_tokenizer", os.path.join(base_path, "latin_tokenizer.py"))
latin_tokenizer = importlib.util.module_from_spec(spec)
loader = importlib.util.LazyLoader(spec.loader)
spec.loader.exec_module(latin_tokenizer)

LatinTokenizer = latin_tokenizer.LatinTokenizer

tokenizer_labert = LatinTokenizer(vocab_file_path, encoder)
# model_labert = AutoModel.from_pretrained(base_path)

model_labert_wsd = AutoModel.from_pretrained(
    "../../labyrinth/data/models/latin-bert-wic-ft",
    output_hidden_states=True,
    output_attentions=False
)

In [3]:
tokenizer_philberta = AutoTokenizer.from_pretrained('bowphs/PhilBerta')
model_philberta = AutoModelForMaskedLM.from_pretrained('bowphs/PhilBerta')

In [4]:
def read_hits(fname):
    try:
        path = os.path.join(outdir, fname)
        instances = pd.read_parquet(path)
    except:
        instances = pd.DataFrame()
    return instances

In [5]:
instances = read_hits("seminarium.parquet")
len(instances)

3315

In [25]:
instances.sample(10, random_state=42)

,target_phrase,target_from,matched_by,target_lemmata,target_token_ids,target_kwic_idx,target_sent_idx,grela_id,target_sentence_id,start_sentence_id,end_sentence_id,kwic_text,kwic_tokens,target_sentence_text,target_sentence_tokens,author,title,not_before,not_after
2425,[Seminario],canonical,lemma,[seminarium],[388909755],[10],[17],noscemus_868519,noscemus_868519_3511,noscemus_868519_3511,noscemus_868519_3512,", quod in cibis ueniat , Ca rolus Stephanus in...","[{'token_id': 388909745, 'token_text': ',', 'l...","appellatur que id oleum po stea cibarium , quo...","[{'token_id': 388909738, 'token_text': 'appell...","Kyber, David",Lexicon rei herbariae trilingue ex variis et o...,1553.0,1553.0
1803,[Seminarium],canonical,lemma,[seminarium],[124960411],[10],[10],cc_13985,cc_13985_123502,cc_13985_123502,cc_13985_123503,Magnum anno insequenti Praesul exorsus aedific...,"[{'token_id': 124960401, 'token_text': 'Magnum...",Magnum anno insequenti Praesul exorsus aedific...,"[{'token_id': 124960401, 'token_text': 'Magnum...","Orlandini, Nicola",Historia Societatis Jesu,1554.0,1606.0
240,[seminarium],canonical,lemma,[seminarium],[249295690],[10],[3],cc_12133,cc_12133_19797,cc_12133_19796,cc_12133_19797,suis in caelo amiserat amplius non obtinerent ...,"[{'token_id': 249295680, 'token_text': 'suis',...",Sed Deus Omnipotens seminarium generis humani ...,"[{'token_id': 249295687, 'token_text': 'Sed', ...",Moneta Cremonensis,Adversus Catharos et Valdenses c.1240,1240.0,1240.0
2553,[seminarium],canonical,lemma,[seminarium],[33374675],[10],[7],cc_13980,cc_13980_1740,cc_13980_1739,cc_13980_1740,"piscium resp . damnatur ac proscribitur , tamq...","[{'token_id': 33374665, 'token_text': 'piscium...","damnatur ac proscribitur , tamquam uitiorum hu...","[{'token_id': 33374668, 'token_text': 'damnatu...","Masenius, Jacobus",Exercitationes Oratoriae,1606.0,1681.0
1029,[seminarium],canonical,lemma,[seminarium],[340397536],[10],[10],noscemus_841471,noscemus_841471_31631,noscemus_841471_31631,noscemus_841471_31631,Ab Unitate nempe cuncta procreantur hsfe omnum...,"[{'token_id': 340397526, 'token_text': 'Ab', '...",Ab Unitate nempe cuncta procreantur hsfe omnum...,"[{'token_id': 340397526, 'token_text': 'Ab', '...","Benedetti, Giovanni Battista",Ioannis Baptistae Benedicti Patritii Veneti Ph...,1585.0,1585.0
2376,[seminario],canonical,lemma,[seminarium],[283613424],[10],[14],noscemus_702149,noscemus_702149_146,noscemus_702149_146,noscemus_702149_147,& ex malo nutrimento humoribus malignis scaten...,"[{'token_id': 283613414, 'token_text': '&', 'l...",Ut enim corpora impura & ex malo nutrimento hu...,"[{'token_id': 283613410, 'token_text': 'Ut', '...","Crato, Johannes","Assertio Iohannis Cratonis a Kraftheim, trium ...",1585.0,1585.0
527,[seminaria],canonical,lemma,[seminarium],[259749883],[10],[11],emlap_100100,emlap_100100_24139,emlap_100100_24139,emlap_100100_24140,"natae sunt , irritare deinceps eundem Archeum ...","[{'token_id': 259749873, 'token_text': 'natae'...","quae natae sunt , irritare deinceps eundem Arc...","[{'token_id': 259749872, 'token_text': 'quae',...","Van Helmont, Jan Baptist",Ortus medicinae,1648.0,1648.0
1667,[seminarium],canonical,lemma,[seminarium],[69936864],[10],[1],cc_11728,cc_11728_905,cc_11728_904,cc_11728_905,torquet pro cibis folia arborum et herbas agre...,"[{'token_id': 69936854, 'token_text': 'torquet...",Hoc seminarium mendacii ab eis publicatum per ...,"[{'token_id': 69936863, 'token_text': 'Hoc', '...",Auctor incertus,Vita S. Wilhelmi,NaN,NaN
3304,[seminarium],canonical,lemma,[seminarium],[294714548],[10],[8],noscemus_705667,noscemus_705667_4598,noscemus_705667_4597,noscemus_705667_4599,natura . In haec enim cornua unum ex utraque p...,"[{'token_id': 294714538, 'token_text': 'natura...","In haec enim cornua unum ex utraque par""te sem...","[{'token_id': 294714540, 'token_text': 'In', '...","Hemmer, Johann Jakob",Conductorum fulmineorum vim egregiam tribus re...,1790.0,1790.0
3150,[seminarium],canonical,lemma,[seminarium],

In [6]:
import embedding_helpers
importlib.reload(embedding_helpers)
from embedding_helpers import *

In [7]:
sample_instance = instances.iloc[0]

In [10]:
# compute embedding
result = embedding_helpers.embed_emlap_instance(
    instance=sample_instance,
    tokenizer=tokenizer_labert,
    model=model_labert_wsd,
    device="cuda",                # or "cpu"
    layer_idx=11,
    piece_pooling="mean",
    context_lemmatized=True,
    target_lemmatized=False,
    context_pos_filtered=True,
)

In [11]:
# compute embedding
result = embedding_helpers.embed_emlap_instance(
    instance=sample_instance,
    tokenizer=tokenizer_philberta,
    model=model_philberta,
    device="cuda",                # or "cpu"
    layer_idx=11,
    piece_pooling="mean",
    context_lemmatized=True,
    target_lemmatized=True,
    context_pos_filtered=True,
)

In [12]:
case_studies_terms = ["pathos", "passio","seminarium", "mercurius", "labyrinthus"]
for term in case_studies_terms:
    instances = read_hits(f"{term}.parquet")
    if len(instances) > 1000:
        instances = instances.sample(n=1000, random_state=42).reset_index(drop=True)
    for title, model, tokenizer in [
        ("labert_wsd", model_labert_wsd, tokenizer_labert),
        ("philberta", model_philberta, tokenizer_philberta)
    ]:
        for layer in [7,11]:
            emb_col_name = f"emb_{title}_l{layer}"
            instances[emb_col_name] = instances.apply(lambda row: embedding_helpers.embed_emlap_instance(
                instance=row,
                tokenizer=tokenizer,
                model=model,
                device="cuda",                # or "cpu"
                layer_idx=layer,
                piece_pooling="mean",
                context_lemmatized=True,
                target_lemmatized=True,
                context_pos_filtered=True)["embedding"], axis=1)
    instances.to_parquet("../data/large_files/phylosen_instances/{}_emb_sample1000.parquet".format(term))